<a href="https://colab.research.google.com/github/KeithRajbhandari/KRB-HTML/blob/main/Week_3_Assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os

# Set your Kaggle credentials directly
os.environ['KAGGLE_USERNAME'] = "keithrajbhandari"
os.environ['KAGGLE_KEY'] = "KGAT_2157dd9f77d9e8fb39ca6e9149cf8379"

# 1. Install PySpark and Kaggle CLI
!pip install -q kaggle pyspark

# 2. Download the healthcare dataset straight into Colab storage
!mkdir -p /content/healthcare_data
!kaggle datasets download -d rohitrox/healthcare-provider-fraud-detection-analysis -p /content/healthcare_data

# 3. Unzip the downloaded files
!unzip -o /content/healthcare_data/healthcare-provider-fraud-detection-analysis.zip -d /content/healthcare_data

# 4. Initialize the PySpark Session
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("HealthcareFraudDetection") \
    .master("local[*]") \
    .getOrCreate()

# 5. Load CSV files into PySpark DataFrames
df_train_ip = spark.read.csv("/content/healthcare_data/Train_Inpatientdata-1542865627584.csv", header=True, inferSchema=True)
df_train_op = spark.read.csv("/content/healthcare_data/Train_Outpatientdata-1542865627584.csv", header=True, inferSchema=True)
df_train_bd = spark.read.csv("/content/healthcare_data/Train_Beneficiarydata-1542865627584.csv", header=True, inferSchema=True)
df_train    = spark.read.csv("/content/healthcare_data/Train-1542865627584.csv", header=True, inferSchema=True)

# 6. Create Spark SQL temporary views for analytical queries
df_train_ip.createOrReplaceTempView("fact_inpatient")
df_train_op.createOrReplaceTempView("fact_outpatient")
df_train_bd.createOrReplaceTempView("dim_beneficiary")
df_train.createOrReplaceTempView("dim_provider")

print("\n-------------------------------------------------------")
print("SUCCESS: Data downloaded and loaded into PySpark SQL views!")
print("-------------------------------------------------------")

Dataset URL: https://www.kaggle.com/datasets/rohitrox/healthcare-provider-fraud-detection-analysis
License(s): CC0-1.0
100% 25.4M/25.4M [00:00<00:00, 76.9MB/s]

Archive:  /content/healthcare_data/healthcare-provider-fraud-detection-analysis.zip
  inflating: /content/healthcare_data/Test-1542969243754.csv  
  inflating: /content/healthcare_data/Test_Beneficiarydata-1542969243754.csv  
  inflating: /content/healthcare_data/Test_Inpatientdata-1542969243754.csv  
  inflating: /content/healthcare_data/Test_Outpatientdata-1542969243754.csv  
  inflating: /content/healthcare_data/Train-1542865627584.csv  
  inflating: /content/healthcare_data/Train_Beneficiarydata-1542865627584.csv  
  inflating: /content/healthcare_data/Train_Inpatientdata-1542865627584.csv  
  inflating: /content/healthcare_data/Train_Outpatientdata-1542865627584.csv  

-------------------------------------------------------
SUCCESS: Data downloaded and loaded into PySpark SQL views!
----------------------------------------

In [4]:
# ---------------------------------------------------------
# 1. ANALYTICAL QUERY: Provider Cost Analysis (JOIN & AGGREGATION)
# ---------------------------------------------------------
print("Analyzing top 10 most expensive healthcare providers...\n")

provider_analysis = spark.sql("""
    SELECT
        p.Provider,
        p.PotentialFraud,
        COUNT(i.ClaimID) AS total_inpatient_claims,
        ROUND(SUM(i.InscClaimAmtReimbursed), 2) AS total_reimbursed_amt,
        ROUND(AVG(i.InscClaimAmtReimbursed), 2) AS avg_claim_amt
    FROM dim_provider p
    LEFT JOIN fact_inpatient i ON p.Provider = i.Provider
    GROUP BY p.Provider, p.PotentialFraud
    ORDER BY total_reimbursed_amt DESC
""")

# Display the top 10 results
provider_analysis.show(10)

# ---------------------------------------------------------
# 2. CREATING A VIEW: Isolating High-Risk Fraudulent Providers
# ---------------------------------------------------------
# This satisfies the "Views" requirement of your assignment.
# It creates a virtual table of only fraudulent providers with high claim counts.

print("\nCreating a view for High-Risk Providers and displaying the top 5...\n")

spark.sql("""
    CREATE OR REPLACE TEMP VIEW high_risk_providers AS
    SELECT
        p.Provider,
        COUNT(i.ClaimID) AS total_claims,
        ROUND(SUM(i.InscClaimAmtReimbursed), 2) AS total_stolen_funds
    FROM dim_provider p
    JOIN fact_inpatient i ON p.Provider = i.Provider
    WHERE p.PotentialFraud = 'Yes'
    GROUP BY p.Provider
    HAVING COUNT(i.ClaimID) > 50
""")

# Query the new view we just created
high_risk_query = spark.sql("""
    SELECT *
    FROM high_risk_providers
    ORDER BY total_stolen_funds DESC
""")

high_risk_query.show(5)

Analyzing top 10 most expensive healthcare providers...

+--------+--------------+----------------------+--------------------+-------------+
|Provider|PotentialFraud|total_inpatient_claims|total_reimbursed_amt|avg_claim_amt|
+--------+--------------+----------------------+--------------------+-------------+
|PRV52019|           Yes|                   516|             5580870|     10815.64|
|PRV55462|           Yes|                   386|             4260100|     11036.53|
|PRV54367|           Yes|                   322|             3040900|      9443.79|
|PRV53706|           Yes|                   282|             2776000|      9843.97|
|PRV55209|           Yes|                   275|             2756100|     10022.18|
|PRV56560|           Yes|                   248|             2605900|     10507.66|
|PRV55230|           Yes|                   225|             2518350|     11192.67|
|PRV54742|           Yes|                   231|             2499900|     10822.08|
|PRV56416|         

In [5]:
# ---------------------------------------------------------
# 3. TABLE CREATION & INDEXING (PARTITIONING)
# ---------------------------------------------------------
# This satisfies the requirement to show DDL (Table Creation)
# and Indexing (Partitioning for Big Data performance).

print("Creating a partitioned physical table (Big Data Indexing)...\n")

# We are creating a permanent table partitioned by the patient's State.
# If we search for patients in state '10', Spark will only read that specific partition,
# massively speeding up the query (acting as our index).

spark.sql("""
    CREATE TABLE IF NOT EXISTS optimized_beneficiaries
    USING PARQUET
    PARTITIONED BY (State)
    AS SELECT * FROM dim_beneficiary
""")

# Test the "indexed" partitioned table
test_partition = spark.sql("""
    SELECT State, COUNT(*) as total_patients
    FROM optimized_beneficiaries
    GROUP BY State
    ORDER BY total_patients DESC
    LIMIT 5
""")

test_partition.show()
print("SUCCESS: Partitioned table created! Rubric requirements met.")

Creating a partitioned physical table (Big Data Indexing)...

+-----+--------------+
|State|total_patients|
+-----+--------------+
|    5|         12052|
|   10|          9771|
|   45|          8780|
|   33|          8443|
|   39|          6055|
+-----+--------------+

SUCCESS: Partitioned table created! Rubric requirements met.
